# Masking Sites from MCMC

In [1]:
import pyro
import pyro.poutine as poutine
import pyro.distributions as dist
import pyro.distributions.transforms as transforms
import pyro.distributions.constraints as constraints
import pyro.infer as infer
import pyro.optim as optim
import torch

import matplotlib.pyplot as plt

import logging
import math

logging.basicConfig(format='%(message)s', level=logging.INFO)

param_store = pyro.get_param_store()

In [16]:
def model():
    x = pyro.sample("x", dist.Normal(0,1))
    y = pyro.sample("y", dist.Normal(x, x.abs() + 1))
    z = pyro.sample("z", dist.Delta(y))
    w = pyro.sample("w", dist.LogNormal(y, 1/(y.abs() + 1)))
    #print("running! {}".format({"x":x, "y":y, "z":z, "w":w}))

child_blocked = poutine.block(model, hide = ["z","w"])
allchild_blocked = poutine.block(model, hide = ["y","z","w"])
delta_blocked = poutine.block(model, hide=["z"])
parent_blocked = poutine.block(model, hide = ["x","z"])

In [17]:
poutine.trace(parent_blocked).get_trace().nodes

OrderedDict([('_INPUT',
              {'name': '_INPUT', 'type': 'args', 'args': (), 'kwargs': {}}),
             ('y',
              {'type': 'sample',
               'name': 'y',
               'fn': Normal(loc: -1.4780765771865845, scale: 2.478076457977295),
               'is_observed': False,
               'args': (),
               'kwargs': {},
               'value': tensor(-2.3748),
               'infer': {},
               'scale': 1.0,
               'mask': None,
               'cond_indep_stack': (),
               'done': True,
               'stop': False,
               'continuation': None}),
             ('w',
              {'type': 'sample',
               'name': 'w',
               'fn': LogNormal(),
               'is_observed': False,
               'args': (),
               'kwargs': {},
               'value': tensor(0.0634),
               'infer': {},
               'scale': 1.0,
               'mask': None,
               'cond_indep_stack': (),
         

In [18]:
nuts = infer.NUTS(parent_blocked)
mcmc = infer.MCMC(nuts, 1000, 1000)

mcmc.run()

Sample: 100%|██████████| 2000/2000 [30:50,  1.08it/s, step size=2.93e-19, acc. prob=0.572]


In [ ]:
pred = infer.Predictive(model, mcmc.get_samples())
pred()